In [6]:
!pip install ultralytics

In [8]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!ls /content/drive/MyDrive/PCB_Project

leewanching.zip  results


In [5]:
import zipfile


zip_path = "/content/drive/MyDrive/PCB_Project/leewanching.zip"


extract_path = "/content/data"


with zipfile.ZipFile(zip_path, "r") as zip_ref:

    zip_ref.extractall(extract_path)


print("Dataset extracted")

Dataset extracted


In [10]:
!ls /content/data/leewanching

set1  set2  set3


In [ ]:
import torch


print(
    "GPU available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        torch.cuda.get_device_name(0)
    )

In [12]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [15]:
from pathlib import Path
import yaml

BASE = Path("/content/data/leewanching")

class_names = {
    0: "missing_hole",
    1: "mouse_bite",
    2: "open_circuit",
    3: "short",
    4: "spur",
    5: "spurious_copper"
}

for set_name in ["set1", "set2", "set3"]:

    yaml_path = BASE / set_name / f"{set_name}.yaml"

    yaml_content = {
        "path": str(BASE / set_name),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": class_names
    }

    with open(yaml_path, "w") as f:
        yaml.safe_dump(
            yaml_content,
            f,
            sort_keys=False
        )

    print("Fixed:", yaml_path)

Fixed: /content/data/leewanching/set1/set1.yaml
Fixed: /content/data/leewanching/set2/set2.yaml
Fixed: /content/data/leewanching/set3/set3.yaml


In [16]:
!cat /content/data/leewanching/set1/set1.yaml

path: /content/data/leewanching/set1
train: images/train
val: images/val
test: images/test
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper


In [17]:
from pathlib import Path

base = Path("/content/data/leewanching/set1")

print("Train exists:", (base / "images/train").exists())
print("Val exists:",   (base / "images/val").exists())
print("Test exists:",  (base / "images/test").exists())

Train exists: True
Val exists: True
Test exists: True


In [18]:
from pathlib import Path
import shutil


for dataset in ["set1","set2","set3"]:

    for split in ["train","val","test"]:

        img_root = Path(
            f"/content/data/leewanching/{dataset}/images/{split}"
        )


        # find all jpg inside class folders
        images = list(img_root.rglob("*.jpg"))


        for img in images:

            # already flat
            if img.parent == img_root:
                continue


            new_path = img_root / img.name


            shutil.move(
                str(img),
                str(new_path)
            )


        # remove empty class folders
        for folder in img_root.iterdir():

            if folder.is_dir():

                shutil.rmtree(folder)


print("Images flattened")

Images flattened


In [19]:
!find /content/data/leewanching -name "*.cache" -delete

In [20]:
from pathlib import Path


base = Path(
    "/content/data/leewanching/set1"
)


train_images = list(
    (base/"images/train").glob("*.jpg")
)


train_labels = list(
    (base/"labels/train").glob("*.txt")
)


print("Images:", len(train_images))
print("Labels:", len(train_labels))

Images: 481
Labels: 481


In [21]:
print(
    train_labels[0].read_text()
)

1 0.320362 0.166280 0.027516 0.024583
1 0.073506 0.461967 0.016509 0.036178
1 0.708333 0.183673 0.022013 0.036178
1 0.785967 0.429499 0.019261 0.038033
1 0.397602 0.522959 0.026336 0.030148


In [22]:
from pathlib import Path


for dataset in ["set1","set2","set3"]:

    print("\n",dataset)

    for split in ["train","val","test"]:

        img_count = len(
            list(
                Path(
                f"/content/data/leewanching/{dataset}/images/{split}"
                ).glob("*.jpg")
            )
        )


        label_count = len(
            list(
                Path(
                f"/content/data/leewanching/{dataset}/labels/{split}"
                ).glob("*.txt")
            )
        )


        print(
            split,
            "images:",
            img_count,
            "labels:",
            label_count
        )


 set1
train images: 481 labels: 481
val images: 60 labels: 60
test images: 152 labels: 152

 set2
train images: 481 labels: 481
val images: 60 labels: 60
test images: 152 labels: 152

 set3
train images: 481 labels: 481
val images: 60 labels: 60
test images: 152 labels: 152


In [16]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="/content/data/leewanching/set1/set1.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=0,
    seed=42,
    device=0,
    project="/content/drive/MyDrive/PCB_Project/results",
    name="set1"
)

Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/leewanching/set1/set1.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=set1, nbs=64, nms=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x780cc91695c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [13]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/PCB_Project/results/set1/weights/best.pt"
)

In [23]:
metrics = model.val(
    data="/content/data/leewanching/set1/set1.yaml",
    split="test"
)

Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 107.4±46.3 MB/s, size: 1238.1 KB)
val: Scanning /content/data/leewanching/set1/labels/test... 152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 152/152 358.6it/s 0.4s
val: New cache created: /content/data/leewanching/set1/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.3s/it 42.9s
                   all        152        520      0.693      0.699      0.691      0.305
          missing_hole         25         85      0.139      0.905      0.542      0.295
            mouse_bite         25         86      0.947      0.833       0.88      0.425
          open_circuit         26         89      0.835      0.674      0.759      0.361
                 short         25         84      0.

In [24]:
precision = metrics.box.mp

recall = metrics.box.mr

map50 = metrics.box.map50

map5095 = metrics.box.map


f1 = (
    2 * precision * recall
    /
    (precision + recall)
)


print("============================")
print("Lee Wan Ching Set 1 Results")
print("============================")

print("Precision :", precision)

print("Recall    :", recall)

print("F1-score  :", f1)

print("mAP50     :", map50)

print("mAP50-95 :", map5095)

Lee Wan Ching Set 1 Results
Precision : 0.6931361103504966
Recall    : 0.6994408094294826
F1-score  : 0.6962741880641707
mAP50     : 0.6912030393563171
mAP50-95 : 0.30503357146840504


In [ ]:
model = YOLO("yolo11n.pt")


model.train(
    data="/content/data/leewanching/set2/set2.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=0,
    seed=42,
    device=0,
    project="/content/drive/MyDrive/PCB_Project/results",
    name="set2"
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/leewanching/set2/set2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=set2, nbs=64, nms=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b9f61fc4ad0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [ ]:
from ultralytics import YOLO


set2_model = YOLO(
    "/content/drive/MyDrive/PCB_Project/results/set2/weights/best.pt"
)

In [ ]:
set2_metrics = set2_model.val(
    data="/content/data/leewanching/set2/set2.yaml",
    split="test"
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 172.4±19.7 MB/s, size: 964.5 KB)
val: Scanning /content/data/leewanching/set2/labels/test... 152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 152/152 489.4it/s 0.3s
val: New cache created: /content/data/leewanching/set2/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.3s
                   all        152        520      0.736       0.56      0.632      0.293
          missing_hole         25         85      0.372      0.847      0.817      0.463
            mouse_bite         25         86          1      0.763      0.836      0.399
          open_circuit         26         89      0.854       0.46      0.619      0.274
                 short         25         84      0.65

In [ ]:
precision = set2_metrics.box.mp

recall = set2_metrics.box.mr

map50 = set2_metrics.box.map50

map5095 = set2_metrics.box.map


f1 = (
    2 * precision * recall /
    (precision + recall)
)


print("============================")
print("Lee Wan Ching Set 2 Results")
print("============================")

print("Precision :", precision)

print("Recall    :", recall)

print("F1-score  :", f1)

print("mAP50     :", map50)

print("mAP50-95 :", map5095)

Set2 Results
Precision : 0.7363134847974363
Recall    : 0.5604353103208325
F1-score  : 0.636447209975204
mAP50     : 0.6321986185316594
mAP50-95 : 0.29256962833789096


In [16]:
model = YOLO("yolo11n.pt")


model.train(
    data="/content/data/leewanching/set3/set3.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=0,
    seed=42,
    device=0,
    project="/content/drive/MyDrive/PCB_Project/results",
    name="set3"
)

Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/leewanching/set3/set3.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=set3, nbs=64, nms=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79c036e78750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [17]:
from ultralytics import YOLO


set3_model = YOLO(
    "/content/drive/MyDrive/PCB_Project/results/set3/weights/best.pt"
)

In [18]:
set3_metrics = set3_model.val(
    data="/content/data/leewanching/set3/set3.yaml",
    split="test"
)

Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4022.9±453.1 MB/s, size: 931.0 KB)
val: Scanning /content/data/leewanching/set3/labels/test... 152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 152/152 2.0Kit/s 0.1s
val: New cache created: /content/data/leewanching/set3/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1it/s 8.9s
                   all        152        520       0.76      0.639      0.715      0.321
          missing_hole         25         85      0.215      0.894       0.72        0.4
            mouse_bite         25         86      0.748      0.761      0.856      0.422
          open_circuit         26         89      0.961      0.558      0.732      0.351
                 short         25         84      0.69

In [19]:
precision = set3_metrics.box.mp

recall = set3_metrics.box.mr

map50 = set3_metrics.box.map50

map5095 = set3_metrics.box.map


f1 = (
    2 * precision * recall /
    (precision + recall)
)


print("============================")
print("Lee Wan Ching Set 3 Results")
print("============================")

print("Precision :", precision)

print("Recall    :", recall)

print("F1-score  :", f1)

print("mAP50     :", map50)

print("mAP50-95 :", map5095)

Lee Wan Ching Set 3 Results
Precision : 0.7604609916502492
Recall    : 0.6385921037138843
F1-score  : 0.6942186626932627
mAP50     : 0.7149193122816929
mAP50-95 : 0.32139142102804724
